# Load Libraries

In [123]:
import requests
import pandas as pd
import geopandas as gpd

## Data Documentation

For the project we merged 2 data sets:
- data with the name and spatial extent of all Danish municipalities
- data with the number of crimes in all Danish municipalities at years 2007 and 2025

The crime rates dataset were retrieved using the statbank API from table STRAF11, from the statistics denmark website.
Each Danish minicipacipality contains the type of crimes commited and the total number of these crimes, An example of how they are stuctured is shown below:


- All types of Crimes:
    - All criminal law crimes:
        - Sexual
        - Violent
        - Property
        - Other criminal law crimes
    - All Special law crimes

For our spartial analysis, we only used the total number of all crimes commited per municipality.


# Data Collection

### Retrieve the table structure and variables

In [124]:
response = requests.get("https://api.statbank.dk/v1/tableinfo/STRAF11")
for var in response.json()['variables']:
    print(var['id'])

OMRÅDE
OVERTRÆD
Tid


### Retrieve the crime rates dataset from API (2007 and 2025)

In [125]:
url = "https://api.statbank.dk/v1/data"

payload = {
    "table": "STRAF11",
    "format": "CSV",
    "variables": [
        {"code": "Tid", "values": ["2007K1", "2007K2", "2007K3", "2007K4","2025K1", "2025K2", "2025K3", "2025K4"]},
        {"code": "OMRÅDE", "values": ["*"]},   
        {"code": "OVERTRÆD", "values": ["*"]}
    ]
}

response = requests.post(url, json=payload)

if response.status_code == 200:
    with open("./data/crime_rate.csv", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("Crime rate data saved successfully")
else:
    print("Error:", response.status_code, response.text)


Crime rate data saved successfully


In [126]:
# Load the crime rate data into a DataFrame and rename columns
crimes = pd.read_csv("./data/crime_rate.csv", sep=";")
crimes.head()


,TID,OMRÅDE,OVERTRÆD,INDHOLD
0,2007K1,Hele landet,Overtrædelsens art i alt,121167
1,2007K1,Hele landet,Straffelov i alt,104604
2,2007K1,Hele landet,Uoplyst straffelov,0
3,2007K1,Hele landet,Seksualforbrydelser i alt,647
4,2007K1,Hele landet,Blodskam mv.,23


In [127]:
crimes.shape

(77168, 4)

### Load geometries per municipality

In [128]:
geom = gpd.read_file("data2023/municipalities_dk.gpkg")
geom = gpd.GeoDataFrame(geom, geometry="geometry")
geom.head()

,kommunekode,navn,region,municipal_id,geometry
0,0370,Næstved,None,370,"MULTIPOLYGON (((672902.45 6143708.87, 672880.8..."
1,0330,Slagelse,None,330,"MULTIPOLYGON (((654675.25 6154223.21, 654636.5..."
2,0766,Hedensted,None,766,"MULTIPOLYGON (((563379.45 6175110.36, 563346.5..."
3,0510,Haderslev,None,510,"MULTIPOLYGON (((516449.61 6137370.13, 516423.3..."
4,0550,Tønder,None,550,"MULTIPOLYGON (((482562.4 6122027.62, 482507.56..."


# Data Preprocessing

In [129]:
'''
First, we translated the column names and filter the data getting only the rows with the total crimes.
Then we grouped each year quarter into years and calculated the total number of crimes per year. 
For example, we computed the sum of total crimes for 2007 as 2007k1 + 2007k2 + 2007k3 + 2007k4 and we did the same for 2025.
'''

crimes.columns = ["quarterYear", "municipality", "typeOfCrime", "totalCrimes"]
crimes = crimes[crimes["typeOfCrime"].str.contains("Overtrædelsens")]

crimes["year"] = crimes["quarterYear"].str[:4]
crimes = (
    crimes.groupby(["municipality", "year"], as_index=False)["totalCrimes"]
    .sum()
)
crimes = crimes[
    ["municipality", "year", "totalCrimes"]
]
crimes.head()

,municipality,year,totalCrimes
0,Aabenraa,2007,6515
1,Aabenraa,2025,3885
2,Aalborg,2007,17958
3,Aalborg,2025,12989
4,Aarhus,2007,36614


In [130]:
## Kept only the name and geometry columns while translating the name column to English
geom = geom[["navn", "geometry"]]
geom.columns = ["name", "geometry"]

# Explore Datasets

We can see that the crime dataset contains also the 5 regions of denmark but we are not gonna use them for this analysis because we need only the municipalities.


In [131]:
print(f'Number of unique municipalities in crime data: {len(crimes["municipality"].unique())}')
print(f'Number of unique municipalities in geometry data: {len(geom["name"].unique())}')
crime_munis = set(crimes["municipality"].str.strip().str.lower())
geom_munis = set(geom["name"].str.strip().str.lower())
crime_munis - geom_munis

Number of unique municipalities in crime data: 106
Number of unique municipalities in geometry data: 99


{'hele landet',
 'region hovedstaden',
 'region midtjylland',
 'region nordjylland',
 'region sjælland',
 'region syddanmark',
 'uoplyst kommune'}

# Merge Datasets

In [132]:
gdf = geom.merge(crimes, left_on="name", right_on="municipality")
gdf = gdf[["municipality", "year", "totalCrimes", "geometry"]]
gdf.head()

,municipality,year,totalCrimes,geometry
0,Næstved,2007,5792,"MULTIPOLYGON (((672902.45 6143708.87, 672880.8..."
1,Næstved,2025,5821,"MULTIPOLYGON (((672902.45 6143708.87, 672880.8..."
2,Slagelse,2007,6959,"MULTIPOLYGON (((654675.25 6154223.21, 654636.5..."
3,Slagelse,2025,5571,"MULTIPOLYGON (((654675.25 6154223.21, 654636.5..."
4,Hedensted,2007,2415,"MULTIPOLYGON (((563379.45 6175110.36, 563346.5..."


In [133]:
gdf.shape

(198, 4)

## Spatial analysis of crime data (year 2007)

In [134]:
gdf_2007 = gdf[gdf["year"] == "2007"]
gdf_2007.head()

,municipality,year,totalCrimes,geometry
0,Næstved,2007,5792,"MULTIPOLYGON (((672902.45 6143708.87, 672880.8..."
2,Slagelse,2007,6959,"MULTIPOLYGON (((654675.25 6154223.21, 654636.5..."
4,Hedensted,2007,2415,"MULTIPOLYGON (((563379.45 6175110.36, 563346.5..."
6,Haderslev,2007,5034,"MULTIPOLYGON (((516449.61 6137370.13, 516423.3..."
8,Tønder,2007,3134,"MULTIPOLYGON (((482562.4 6122027.62, 482507.56..."


## Spatial analysis of crime data (year 2025)

In [136]:
gdf_2025 = gdf[gdf["year"] == "2025"]
gdf_2025.head()

,municipality,year,totalCrimes,geometry
1,Næstved,2025,5821,"MULTIPOLYGON (((672902.45 6143708.87, 672880.8..."
3,Slagelse,2025,5571,"MULTIPOLYGON (((654675.25 6154223.21, 654636.5..."
5,Hedensted,2025,1914,"MULTIPOLYGON (((563379.45 6175110.36, 563346.5..."
7,Haderslev,2025,3007,"MULTIPOLYGON (((516449.61 6137370.13, 516423.3..."
9,Tønder,2025,1801,"MULTIPOLYGON (((482562.4 6122027.62, 482507.56..."
